# State Reducer
State Reducer是LangGraph中用于状态合并的核心机制。通过Annotated指定Reducer，如果未执行，则默认覆盖（Last-Write-Wins）
- 前置知识Annotated
- 常见内置Reducer
- 自定义Reducer


## 常见内置Reducer
- operator.add
- langgraph.graph.message.add_messages

### 内置Reducer-operator.add
- 使用见[01_图的构建与运行.ipynb](./01_图的构建与运行.ipynb)中的`OverAllState`的`logs`字段，在节点执行完后，会调用`add`对字段做合并。

### 内置add_messages
- 对`BaseMessage`列表进行合并(相同消息更新，不同消息添加)，用于汇总一个历史消息列表
- 核心逻辑：
    - 每个`BaseMessage`都有一个id，入参为左右两个`BaseMessage`列表，会判断右边`BaseMessage`列表中的每一项的id是否存在于左边`BaseMessage`列表,如果存在，修改；如果不存在，添加到列表末尾。

In [1]:
from typing import TypedDict, Annotated

from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages


left_messages = [
    SystemMessage(content="你是个善解人意的助手", id='1'),
    HumanMessage(content="你好", id='2'),
    AIMessage(content="你好~", id='3'),
]

right_messages = [
    HumanMessage(content="我是老王，你是小王", id='2'),
    AIMessage(content="好的，我记住啦", id='3'),
    HumanMessage(content="你是谁？", id='4'),
    AIMessage(content="我是小王", id='5'),
]

merged = add_messages(left_messages, right_messages)

for msg in merged:
    print(msg)

content='你是个善解人意的助手' additional_kwargs={} response_metadata={} id='1'
content='我是老王，你是小王' additional_kwargs={} response_metadata={} id='2'
content='好的，我记住啦' additional_kwargs={} response_metadata={} id='3' tool_calls=[] invalid_tool_calls=[]
content='你是谁？' additional_kwargs={} response_metadata={} id='4'
content='我是小王' additional_kwargs={} response_metadata={} id='5' tool_calls=[] invalid_tool_calls=[]


## 自定义State Reducer
- 2个入参，返回合并结构
- 声明如下，具体执行逻辑是在Node执行完时调用reducer逻辑，具体可参考[01_图的构建与运行.ipynb](01_图的构建与运行.ipynb)中的`add`

In [ ]:
from operator import add


def my_reducer(left: list[str], right: list[str]) -> list[str]:
    return left + right

class StateSchemaCloud(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str
    info: Annotated[str, my_reducer]